# SpecMind E trên Google Colab

E bổ sung các dòng code lấy từ dynamic slicing vào prompt. Chọn runtime CPU, bật secret OPENROUTER_API_KEY rồi chọn Runtime → Run all. Đặt ZIP đầu vào tại MyDrive/specmind/inputs/marshmallow-inputs.zip.

Full chạy 24 public API × M1/M2, tối đa 12 attempt mỗi tổ hợp. Pilot chạy Schema.dumps × M1 với một attempt. Mỗi tổ hợp lưu phản hồi, bằng chứng và kết quả trên Drive. Chọn RUN_NAME riêng khi thay model.


In [ ]:
MODE = "full"  # @param ["pilot", "full"]
RUN_NAME = "marshmallow-E"  # @param {type:"string"}
MODEL = "openai/gpt-5.6-luna"  # @param {type:"string"}

import hashlib, json, os, re, shutil, subprocess, sys, zipfile
from pathlib import Path
from google.colab import drive, userdata

if MODE not in ("pilot", "full") or not re.fullmatch(r"[a-zA-Z0-9_-]{1,64}", RUN_NAME):
    raise ValueError("Kiểm tra MODE và RUN_NAME.")
drive.mount("/content/drive")
home = Path("/content/drive/MyDrive/specmind")
run_name = RUN_NAME + "-E-" + MODE
dest = home / "runs" / run_name
root = Path("/content/specmind-work") / run_name
root.mkdir(parents=True, exist_ok=True)

### Tải ZIP từ Drive vào Colab
Notebook kiểm tra SHA-256 rồi giải nén vào ổ tạm của Colab.

In [ ]:
BUNDLE_SHA = "57b94b6bd725dd461f16209c6680611f0173d8ed0b7afd662fd3cbab22a1955d"
saved = home / "inputs" / "marshmallow-inputs.zip"
if not saved.is_file():
    raise FileNotFoundError(f"Hãy đặt file ZIP đầu vào tại {saved}")
local_zip = root / "inputs.zip"
shutil.copyfile(saved, local_zip)
if hashlib.sha256(local_zip.read_bytes()).hexdigest() != BUNDLE_SHA:
    raise ValueError("ZIP trên Drive khác bộ dữ liệu Marshmallow đã chọn.")
inputs = root / "inputs"
with zipfile.ZipFile(local_zip) as z:
    z.extractall(inputs)
print("Đã giải nén dataset, M1, M2 và call graph trên máy Colab.")

### Tải hai codebase và cài môi trường Python 3.11
SpecMind chạy evaluator. Marshmallow là project chứa các hàm và test. Cả hai checkout đều được ghim theo commit.

In [ ]:
def run(args, **kwargs):
    subprocess.run([str(x) for x in args], check=True, **kwargs)

def checkout(name, url, commit):
    path = root / name
    if not (path / ".git").exists():
        run(["git", "init", path])
        run(["git", "-C", path, "remote", "add", "origin", url])
    actual = subprocess.run(["git", "-C", str(path), "rev-parse", "HEAD"], capture_output=True, text=True)
    if actual.stdout.strip() != commit:
        run(["git", "-C", path, "fetch", "--depth=1", "origin", commit])
        run(["git", "-C", path, "checkout", "--detach", commit])
    run(["git", "-C", path, "diff", "--quiet", commit])
    return path

codebase = checkout("engine", "https://github.com/TienMinh-08i/spm-.git", "7edcfa323409dac1be87137735e3433a1e704f85")
repo = checkout("marshmallow", "https://github.com/marshmallow-code/marshmallow.git", "60a803821466f3e537fa2729d65e1aaf7df9d2a4")
run([sys.executable, "-m", "pip", "install", "-q", "uv"])
uv = [sys.executable, "-m", "uv"]
py = root / "venv/bin/python"
if not py.is_file():
    run([*uv, "venv", "--python", "3.11", "--python-preference", "only-managed", root/"venv"])
project = json.loads((inputs/"dataset/manifest.json").read_text())["project"]
run([*uv, "pip", "install", "--python", py, codebase, *project["dependencies"]])
run([py, "-c", "import specmind_eval.run, specmind_prep.mutant_filter, pytest, simplejson"])

### Chạy và lưu kết quả
Pytest ghi file tạm trên máy Colab. Mỗi phản hồi API được lưu ngay trên Drive. Khi xong một tổ hợp, notebook nén bằng chứng và ghi kết quả riêng. Có thể mở lại cùng cấu hình và bấm Run all để tiếp tục nếu môi trường khớp. Nếu yêu cầu API dở dang thiếu phản hồi đã lưu, notebook dừng để tránh tự gọi trùng.

In [ ]:
# @title Bộ chạy SpecMind và lưu checkpoint
worker = root / 'run_specmind.py'
worker.write_text(r'''"""Chạy bằng Python trong venv của SpecMind."""
import csv
import importlib
import json
import os
import shutil
import sys
import zipfile
from pathlib import Path

from openai import OpenAI
from specmind_eval.common.fingerprint import digest, file_hash, source_environment
from specmind_eval.common.storage import read, write
from specmind_eval.config import DEFAULTS
from specmind_eval.context.slicing import load_slice_context
from specmind_eval.generate.loop import generate
from specmind_eval.inputs.dataset import pytest_nodeid
from specmind_eval.inputs.prepared import load_prepared_dataset, load_prepared_mutants
from specmind_prep.filter.dataset import prepare_dataset
from specmind_prep.mutant_filter import filter_mutants
from specmind_prep.scope.scope import prepare_scope, load_scope

cfg = read(sys.argv[1])
root, drive, repo, inputs = map(Path, (cfg[k] for k in ('root','drive','repo','inputs')))
engine = importlib.import_module('specmind_eval.run')
project = read(inputs/'dataset/manifest.json')['project']
problems = [json.loads(s) for s in (inputs/'dataset/problems.jsonl').read_text().splitlines()]
expected_tests = {}
for line in (inputs/'dataset/tests.jsonl').read_text().splitlines():
    row = json.loads(line)
    expected_tests.setdefault(row['problem_id'], set()).add(pytest_nodeid(row))
tasks = [p['function_id'] for p in problems] if cfg['full'] else ['marshmallow.schema.Schema.dumps']
corpora = ('M1','M2') if cfg['full'] else ('M1',)
arms = {'E': 'slice'}
load_slice_context(cfg['slices'], repo, problems, 16000)
identity = digest([cfg, source_environment(repo), file_hash(__file__)])
drive.mkdir(parents=True, exist_ok=True)
if (drive/'run.json').exists() and read(drive/'run.json')['identity'] != identity:
    raise ValueError('Cấu hình hoặc runtime khác. Chọn RUN_NAME mới để giữ nguyên kết quả cũ.')
write(drive/'run.json', {'identity':identity,'config':cfg,'tasks':tasks})
client = OpenAI(api_key=os.environ.pop('SPECMIND_COLAB_KEY'), base_url='https://openrouter.ai/api/v1',
                max_retries=0, timeout=180)

def archive(directory, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = shutil.make_archive(str(directory)+'.snapshot', 'zip', root_dir=directory)
    part = target.with_suffix('.partial')
    shutil.copyfile(temp, part)
    if file_hash(part) != file_hash(temp):
        raise OSError('Drive checksum mismatch: '+str(part))
    part.replace(target)
    Path(temp).unlink()

def restore(archive_path, directory):
    with zipfile.ZipFile(archive_path) as z:
        for name in z.namelist():
            if not (directory/name).resolve().is_relative_to(directory.resolve()):
                raise ValueError('Unsafe archive path')
        z.extractall(directory)

def prepare(task, work, corpus):
    ds, scope = work/'dataset', work/'scope'
    if not (ds/'manifest.json').exists() or not read(ds/'manifest.json')['complete']:
        prepare_dataset(inputs/'dataset/problems.jsonl', inputs/'dataset/tests.jsonl', repo, ds,
                        resume=True, tasks=[task], project=project, timeout=300,
                        lineage={'input_zip_sha256':cfg['bundle']})
    selected, tests, dm = load_prepared_dataset(ds, repo)
    if len(selected)!=1 or {pytest_nodeid(t) for t in tests.get(task, [])} != expected_tests[task]:
        raise ValueError('Bộ test Linux khác bộ Windows: '+task)
    prepare_scope(ds, repo, scope, resume=True)
    scopes, targets, sm = load_scope(scope, repo)
    target = targets[0]
    source_dir = inputs/corpus
    parent = read(source_dir/'manifest.json')
    row, = [r for r in read(source_dir/'functions.json') if r['function']==task]
    if row['analysis_target'] != target['problem_id'] or selected[0]['source'] != target['source']:
        raise ValueError('Target/source mismatch: '+task)
    for m in row['mutants']:
        if m['mutation_problem']['source'] != target['source'] or m['mutation_target'] != target['problem_id']:
            raise ValueError('Mutant target mismatch: '+m['id'])
    seed = work/(corpus+'-seed')
    write(seed/'functions.json', [dict(row, scope=scopes[0])])
    write(seed/'targets_input.json', targets)
    write(seed/'summary.json', {'functions':1,'task_mutant_links':len(row['mutants'])})
    manifest = dict(schema='prepared-mutants-v1', complete=True, generator=parent['generator'],
                    method=corpus, project=project, runtime_filter_required=True,
                    dataset_fingerprint=dm['fingerprint'], dataset_files=dm['files'],
                    scope_fingerprint=sm['fingerprint'], scope_files=sm['files'],
                    source_environment=dm['source_environment'], imported_from=parent['fingerprint'],
                    files={f:file_hash(seed/f) for f in ('functions.json','targets_input.json','summary.json')})
    write(seed/'manifest.json', dict(manifest, fingerprint=digest(manifest)))
    filtered = work/corpus
    filter_mutants(ds, seed, repo, filtered, timeout=300, workers=1, resume=True)
    actual, _ = load_prepared_mutants(filtered, repo, dm)
    if [m['id'] for m in actual[task]['mutants']] != [m['id'] for m in row['mutants']]:
        raise ValueError('Bộ mutant Linux khác bộ Windows, xem excluded_mutants.json: '+task)
    return ds, filtered

def evaluate(task, corpus, arm, mode, ds, mutants, directory, name):
    settings = dict(DEFAULTS, tasks=[task], repo_root=str(repo), expected_repo_commit=project['commit'],
                    prepared_dataset=str(ds), prepared_mutants=str(mutants), model=cfg['model'],
                    max_turns=cfg['turns'], max_tokens=4096, api_retries=0,
                    generate_without_mutants=True, context_mode=mode, caller_context='full',
                    context_budget=16000, slice_context_dir=cfg['slices'],
                    pytest_timeout=300, batch_size=500,
                    state_attributes={'marshmallow.decorators.set_hook':['__marshmallow_hook__']})
    def request(messages, config):
        # Count occurrences so identical prompts in different turns remain independent.
        nonlocal turn
        turn += 1
        key = digest([identity, name, turn, messages])
        path = drive/'responses'/name/f'turn-{turn:02}.json'
        if path.exists():
            saved = read(path)
            if saved['request_sha256'] != key:
                raise ValueError('Lịch sử prompt khi chạy lại khác checkpoint: '+str(path))
            raw = saved['response']
            return raw['choices'][0]['message']['content'], dict(raw.get('usage') or {}, cached_response=True)
        marker = path.with_suffix('.started')
        marker.parent.mkdir(parents=True, exist_ok=True)
        with marker.open('x') as f:
            f.write(key)
        raw = client.chat.completions.create(model=config['model'], messages=messages,
                    temperature=0, max_tokens=config['max_tokens']).model_dump(mode='json')
        write(path, {'request_sha256':key, 'response':raw})
        return raw['choices'][0]['message']['content'], raw.get('usage') or {}
    turn = 0
    engine.generate = lambda *a, **kw: generate(*a, **kw, call=request)
    engine.preflight(settings, directory/'preflight')
    # Rebuild interrupted task from its durable responses in exactly the same turn order.
    engine.run(settings, directory/'evaluation')
    result, = read(directory/'evaluation/results.json')
    if not result.get('generation_done') or result['status'] not in ('completed','evaluation_pending'):
        raise RuntimeError(str(result.get('error') or result['status']))
    return result

rows = []
for task in tasks:
    for corpus in corpora:
        name = corpus+'_'+task
        work = root/'prepared'/name
        prep_zip = drive/'prepared'/(name+'.zip')
        todo = [a for a in arms if not (drive/'completed'/(name+'_'+a+'.json')).exists()]
        if todo:
            if prep_zip.exists() and not work.exists():
                restore(prep_zip, work)
            try:
                ds, mutants = prepare(task, work, corpus)
            except Exception:
                if work.exists():
                    archive(work, drive/'diagnostics'/(name+'.zip'))
                raise
            if not prep_zip.exists():
                archive(work, prep_zip)
        for arm, mode in arms.items():
            unit = name+'_'+arm
            receipt = drive/'completed'/(unit+'.json')
            evidence = drive/'runs'/(unit+'.zip')
            if not receipt.exists():
                import tempfile
                folder = Path(tempfile.mkdtemp(dir=root))
                print('Đang chạy', task, corpus, arm, flush=True)
                try:
                    result = evaluate(task, corpus, arm, mode, ds, mutants, folder, unit)
                finally:
                    archive(folder, evidence)
                write(receipt, dict(task=task, corpus=corpus, arm=arm, status=result['status'],
                                    attempts=result['attempts'], correctness=(result.get('correctness') or {}).get('status'),
                                    **((result.get('mutation') or {}).get('metrics',{}).get('feedback') or {})))
                shutil.rmtree(folder)
            elif not evidence.is_file():
                raise FileNotFoundError(evidence)
            rows.append(read(receipt))
        if work.exists():
            shutil.rmtree(work)
columns = list(dict.fromkeys(k for row in rows for k in row))
with (drive/'RESULTS.tmp').open('w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    writer.writerows(rows)
(drive/'RESULTS.tmp').replace(drive/'RESULTS.csv')
print('Đã lưu',len(rows),'kết quả tại',drive,flush=True)
''', encoding='utf-8')

In [ ]:
settings = dict(engine_commit='7edcfa323409dac1be87137735e3433a1e704f85', slices=str(codebase/'experiments/marshmallow_e'), root=str(root), drive=str(dest), repo=str(repo), inputs=str(inputs), bundle=BUNDLE_SHA,
                full=MODE=="full", model=MODEL, turns=12 if MODE=="full" else 1)
config = root / "settings.json"
config.write_text(json.dumps(settings), encoding="utf-8")
subprocess.run([str(py), "-u", str(worker), str(config)], cwd=root, check=True,
               env={**os.environ, "PYTHONPATH":"", "PYTHONNOUSERSITE":"1", "PYTEST_DISABLE_PLUGIN_AUTOLOAD":"1",
                    "SPECMIND_COLAB_KEY":userdata.get("OPENROUTER_API_KEY").strip()})

In [ ]:
import pandas as pd
display(pd.read_csv(dest / "RESULTS.csv"))
print("Kết quả trên Google Drive:", dest)